In [ ]:
from pathlib import Path
import subprocess
import json
from datetime import datetime

# ============================================================
# Sync local conda environments from environment.yml files
#
# Folder structure expected:
#
# project_root/
#   Meta Scripts/
#     sync_conda_environments.ipynb
#   environments/
#     cellpose/
#       environment.yml
#     yolo-seg/
#       environment.yml
#     update_environment/
#       environment.yml
#
# Each folder name should match the conda environment name.
# ============================================================

DRY_RUN = False
PRUNE_EXTRA_PACKAGES = True

cwd = Path.cwd()

if cwd.name == "Meta Scripts":
  project_root = cwd.parent
else:
  project_root = cwd

environments_dir = project_root / "environments"

if not environments_dir.exists():
  raise FileNotFoundError(f"Could not find environments folder at: {environments_dir}")

print(f"Project root: {project_root}")
print(f"Environments folder: {environments_dir}")


def run_command(command):
  print("\nRunning:")
  print(" ".join(str(part) for part in command))

  if DRY_RUN:
    print("DRY_RUN is True, so this command was not executed.")
    return ""

  result = subprocess.run(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    shell=False
  )

  if result.returncode != 0:
    raise RuntimeError(
      "Command failed:\n"
      + " ".join(str(part) for part in command)
      + "\n\nSTDOUT:\n"
      + result.stdout
      + "\n\nSTDERR:\n"
      + result.stderr
    )

  if result.stdout.strip():
    print(result.stdout)

  if result.stderr.strip():
    print(result.stderr)

  return result.stdout


def get_available_conda_env_names():
  conda_info_raw = run_command(["conda", "env", "list", "--json"])
  conda_info = json.loads(conda_info_raw)

  env_paths = [Path(path) for path in conda_info.get("envs", [])]
  env_names = {path.name for path in env_paths}

  return env_names


available_env_names = get_available_conda_env_names()

print("\nCurrently available conda environments:")
for env_name in sorted(available_env_names):
  print(f"  - {env_name}")


env_folders = [
  folder for folder in environments_dir.iterdir()
  if folder.is_dir()
]

if len(env_folders) == 0:
  print("\nNo environment folders found.")
else:
  print("\nEnvironment folders found:")
  for folder in env_folders:
    print(f"  - {folder.name}")


timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
summary = []

for env_folder in env_folders:
  env_name = env_folder.name
  environment_yml_path = env_folder / "environment.yml"

  print("\n" + "=" * 80)
  print(f"Checking environment folder: {env_name}")

  if not environment_yml_path.exists():
    print(f"Skipping '{env_name}': no environment.yml found.")
    summary.append((env_name, "skipped - no environment.yml"))
    continue

  if env_name in available_env_names:
    print(f"Conda environment '{env_name}' already exists.")
    print("Updating it from environment.yml...")

    command = [
      "conda", "env", "update",
      "-n", env_name,
      "-f", str(environment_yml_path)
    ]

    if PRUNE_EXTRA_PACKAGES:
      command.append("--prune")

    run_command(command)

    summary.append((env_name, "updated"))

  else:
    print(f"Conda environment '{env_name}' does not exist.")
    print("Creating it from environment.yml...")

    run_command([
      "conda", "env", "create",
      "-n", env_name,
      "-f", str(environment_yml_path)
    ])

    available_env_names.add(env_name)
    summary.append((env_name, "created"))


print("\n" + "=" * 80)
print("Sync summary:")
print(f"Finished at: {timestamp}")

for env_name, status in summary:
  print(f"  - {env_name}: {status}")

print("\nDone.")